In [7]:
import polars as pl
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

os.makedirs("models", exist_ok=True)

# 1. DATA LOADING WITH FALLBACK
file_path = "C:/Users/HP/Downloads/student_por.csv"
try:
    df = pl.read_csv(file_path, separator=";")
    print("Real data loaded successfully!")
except Exception:
    print("Data")
    df = pl.DataFrame({
        "studytime": np.random.randint(1, 5, 100),
        "absences": np.random.randint(0, 20, 100),
        "sex": np.random.choice(["M", "F"], 100),
        "G3": np.random.randint(0, 20, 100)
    })

# 2. PROCESSING
df = df.with_columns((pl.col("G3") >= 10).cast(pl.Int64).alias("Pass"))
le = LabelEncoder()
df = df.with_columns(pl.Series("sex", le.fit_transform(df["sex"].to_numpy())))

features = ["studytime", "absences", "sex"]
X = df.select(features).to_numpy()
y = df["Pass"].to_numpy()

# 3. SELECTION & SPLIT
X = SelectKBest(score_func=f_classif, k=3).fit_transform(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. SCALING
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
joblib.dump(scaler, "models/scaler.pkl")

# 5. TRAINING
models = {
    "Logistic": LogisticRegression(),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} Acc: {accuracy_score(y_test, model.predict(X_test)):.2f}")

# 6. TUNING
grid = GridSearchCV(RandomForestClassifier(), {"n_estimators": [50, 100], "max_depth": [3, 5]}, cv=2)
grid.fit(X_train, y_train)
joblib.dump(grid.best_estimator_, "models/model.pkl")

# 7. PREDICTION FUNCTION
def predict_student(data_row):
    model = joblib.load("models/model.pkl")
    scaler = joblib.load("models/scaler.pkl")
    prediction = model.predict(scaler.transform(data_row))
    return "PASS" if prediction[0] == 1 else "FAIL"

print("\nSample Prediction:", predict_student(X_test[0:1]))

Data
Logistic Acc: 0.60
RandomForest Acc: 0.50
XGBoost Acc: 0.40

Sample Prediction: PASS
